In [1]:
# -*- coding: utf-8 -*-
# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: light
#       format_version: '1.5'
#       jupytext_version: 1.14.5
#   kernelspec:
#     display_name: Python 3 (ipykernel)
#     language: python
#     name: python3
# ---

# # Cybersecurity Named Entity Recognition (NER) Inference
# 
# This notebook allows you to use your trained BEACON models to perform inference on new text.

# ## 1. Setup and Imports

import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils import data
from transformers import AutoTokenizer, AutoModel, BertModel, RobertaModel
from typing import List, Dict, Tuple, Optional
import json
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML, Markdown

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ## 2. Define Model Architecture & Required Classes

class InputExample:
    """A single example for token classification."""
    def __init__(self, guid, words, labels, source="inference"):
        self.guid = guid
        self.words = words
        self.labels = labels
        self.source = source

class InputFeatures:
    """Features created from a single example."""
    def __init__(self, input_ids, input_mask, segment_ids, predict_mask, label_ids):
        self.input_ids = input_ids
        self.input_mask = input_mask
        self.segment_ids = segment_ids
        self.predict_mask = predict_mask  # Mask for the first subword of each original word
        self.label_ids = label_ids

def example2feature(example, tokenizer, label_map, max_seq_length):
    """Converts a single `InputExample` into an `InputFeatures`."""
    add_label = 'X'  # Label for subsequent subword tokens
    tokens = []
    label_ids = []
    predict_mask = []  # 1 for the first subword token of a word, 0 otherwise

    # [CLS] token
    tokens.append(tokenizer.cls_token)
    label_ids.append(label_map.get('[CLS]', label_map['O']))
    predict_mask.append(0)  # Do not predict for [CLS]

    # Word tokens
    for i, word in enumerate(example.words):
        word_tokenized = tokenizer.tokenize(str(word))
        if not word_tokenized:
            word_tokenized = [tokenizer.unk_token]

        tokens.extend(word_tokenized)

        # Assign label to the *first* subword token, 'X' to others
        label = example.labels[i] if i < len(example.labels) else 'O'
        for j, sub_token in enumerate(word_tokenized):
            if j == 0:
                label_ids.append(label_map.get(label, label_map['O']))
                predict_mask.append(1)  # Predict for this token
            else:
                label_ids.append(label_map.get(add_label, label_map['O']))
                predict_mask.append(0)  # Do not predict for subsequent subwords

    # Truncation
    if len(tokens) > max_seq_length - 1:  # -1 for [SEP]
        tokens = tokens[:(max_seq_length - 1)]
        label_ids = label_ids[:(max_seq_length - 1)]
        predict_mask = predict_mask[:(max_seq_length - 1)]

    # [SEP] token
    tokens.append(tokenizer.sep_token)
    label_ids.append(label_map.get('[SEP]', label_map['O']))
    predict_mask.append(0)  # Do not predict for [SEP]

    # Convert tokens to ids
    input_ids = tokenizer.convert_tokens_to_ids(tokens)
    input_mask = [1] * len(input_ids)
    segment_ids = [0] * len(input_ids)

    return InputFeatures(
        input_ids=input_ids,
        input_mask=input_mask,
        segment_ids=segment_ids,
        predict_mask=predict_mask,
        label_ids=label_ids
    )

class NerDataset(data.Dataset):
    """Dataset for NER examples."""
    def __init__(self, examples, tokenizer, label_map, max_seq_length):
        self.examples = examples
        self.tokenizer = tokenizer
        self.label_map = label_map
        self.max_seq_length = max_seq_length
        self.features = self._create_features()

    def _create_features(self):
        features = []
        for example in self.examples:
            feat = example2feature(example, self.tokenizer, self.label_map, self.max_seq_length)
            if feat is not None:  # Check if feature creation was successful
                features.append(feat)
        return features

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        feat = self.features[idx]
        return (
            feat.input_ids,
            feat.input_mask,
            feat.segment_ids,
            feat.predict_mask,
            feat.label_ids
        )

    @staticmethod
    def pad(batch):
        """Pads sequences within a batch to the maximum sequence length."""
        seqlen_list = [len(sample[0]) for sample in batch]
        maxlen = max(seqlen_list) if seqlen_list else 0
        pad_label_id = 0  # Default padding ID for labels

        f = lambda x, seqlen, pad_value: [sample[x] + [pad_value] * (seqlen - len(sample[x])) for sample in batch]

        input_ids_list = torch.LongTensor(f(0, maxlen, 0))
        input_mask_list = torch.LongTensor(f(1, maxlen, 0))
        segment_ids_list = torch.LongTensor(f(2, maxlen, 0))
        predict_mask_int = f(3, maxlen, 0)
        predict_mask_list = torch.BoolTensor(predict_mask_int)
        label_ids_list = torch.LongTensor(f(4, maxlen, pad_label_id))

        return input_ids_list, input_mask_list, segment_ids_list, predict_mask_list, label_ids_list

def log_sum_exp_batch(log_tensor, axis=-1):
    """Calculates log_sum_exp in a numerically stable way for a batch."""
    if log_tensor.nelement() == 0:
        out_shape = list(log_tensor.shape)
        if axis is not None and 0 <= axis < len(out_shape): del out_shape[axis]
        elif axis is not None and axis < 0 and abs(axis) <= len(out_shape): del out_shape[axis]
        return torch.full(out_shape, -float('inf'), device=log_tensor.device, dtype=log_tensor.dtype)

    max_score = torch.max(log_tensor, axis, keepdim=True)[0]
    is_inf = torch.isneginf(max_score)
    max_score_adjusted = torch.where(is_inf, torch.zeros_like(max_score), max_score)
    sum_exp = torch.exp(log_tensor - max_score_adjusted).sum(axis, keepdim=True)
    log_sum_exp_val = torch.where(sum_exp > 0, torch.log(sum_exp) + max_score_adjusted, torch.full_like(max_score_adjusted, -float('inf')))
    log_sum_exp_val = torch.where(is_inf, torch.full_like(log_sum_exp_val, -float('inf')), log_sum_exp_val)
    return log_sum_exp_val.squeeze(axis)

class BERT_CRF_NER(nn.Module):
    """BERT-CRF model for NER."""
    def __init__(self, model_type, start_label_id, stop_label_id, num_labels, device, hf_token=None):
        super(BERT_CRF_NER, self).__init__()
        self.num_labels = num_labels
        self.start_label_id = start_label_id
        self.stop_label_id = stop_label_id
        self.device = device
        self.model_type = model_type
        self.hf_token = hf_token

        # Initialize encoder based on model type
        try:
            print(f"Initializing encoder: {model_type}")
            model_type_lower = model_type.lower()
            
            # Handle different model types
            if "roberta" in model_type_lower:
                self.encoder = AutoModel.from_pretrained(model_type, token=self.hf_token)
            elif "securebert" in model_type_lower:
                self.encoder = AutoModel.from_pretrained("ehsanaghaei/SecureBERT", token=self.hf_token)
            elif "darkbert" in model_type_lower:
                self.encoder = AutoModel.from_pretrained("s2w-ai/DarkBERT", token=self.hf_token)
            elif "cysecbert" in model_type_lower:
                self.encoder = AutoModel.from_pretrained("markusbayer/CySecBERT", token=self.hf_token)
            else:  # Default to BERT
                self.encoder = AutoModel.from_pretrained(model_type, token=self.hf_token)
                
        except Exception as e:
            print(f"Error loading model '{model_type}' from Hugging Face: {e}")
            raise

        self.hidden_size = self.encoder.config.hidden_size
        
        # Classifier and Dropout
        self.dropout = nn.Dropout(0.2)
        self.hidden2label = nn.Linear(self.hidden_size, self.num_labels)

        # CRF Layer
        self.transitions = nn.Parameter(torch.randn(self.num_labels, self.num_labels))
        # Constrain transitions
        self.transitions.data[start_label_id, :] = -10000.0  # Cannot transition to START
        self.transitions.data[:, stop_label_id] = -10000.0  # Cannot transition from STOP

        # Initialization
        nn.init.xavier_uniform_(self.hidden2label.weight)
        nn.init.constant_(self.hidden2label.bias, 0.0)

    def _get_encoder_features(self, input_ids, segment_ids, input_mask):
        """Get emission scores from the encoder."""
        model_type_lower = self.model_type.lower()
        # RoBERTa-like models don't use token_type_ids (segment_ids)
        if 'roberta' in model_type_lower:
            encoder_output = self.encoder(input_ids=input_ids, attention_mask=input_mask)
        else:  # BERT, SecureBERT, CySecBERT, DarkBERT etc.
            encoder_output = self.encoder(input_ids=input_ids, token_type_ids=segment_ids, attention_mask=input_mask)

        # Handle different output formats
        if hasattr(encoder_output, 'last_hidden_state'):
            sequence_output = encoder_output.last_hidden_state
        elif isinstance(encoder_output, tuple) or isinstance(encoder_output, list):
            sequence_output = encoder_output[0]
        else:
            print("Warning: Unexpected encoder output format. Assuming direct output is sequence output.")
            sequence_output = encoder_output

        sequence_output = self.dropout(sequence_output)
        emission_scores = self.hidden2label(sequence_output)
        return emission_scores

    def _viterbi_decode(self, feats, mask):
        """Find the best path using the Viterbi algorithm."""
        batch_size, seq_len, num_labels = feats.shape
        mask_float = mask.float()
        mask_bool = mask.bool()

        # Initialize Viterbi
        log_delta = torch.full((batch_size, num_labels), -10000.0, device=self.device)
        log_delta[:, self.start_label_id] = 0.0
        psi = torch.zeros((batch_size, seq_len, num_labels), dtype=torch.long, device=self.device)

        transitions_expanded = self.transitions.unsqueeze(0)

        # Forward pass
        for t in range(seq_len):
            mask_t = mask_bool[:, t].unsqueeze(1)
            if not mask_t.any(): continue

            emit_scores_t = feats[:, t]
            scores_t = log_delta.unsqueeze(2) + transitions_expanded
            max_scores_t, max_indices_t = torch.max(scores_t, dim=1)
            max_scores_t += emit_scores_t
            psi[:, t, :] = max_indices_t
            log_delta = torch.where(mask_t, max_scores_t, log_delta)

        # Add final transition to STOP state
        log_delta += self.transitions[:, self.stop_label_id].unsqueeze(0)

        # Backtracking
        best_paths = torch.zeros((batch_size, seq_len), dtype=torch.long, device=self.device)
        best_scores, last_tags = torch.max(log_delta, dim=1)

        for b in range(batch_size):
            seq_len_b = int(mask_float[b].sum().item())
            if seq_len_b == 0: continue
            best_paths[b, seq_len_b - 1] = last_tags[b]
            for t in range(seq_len_b - 2, -1, -1):
                current_tag_idx_at_t_plus_1 = best_paths[b, t + 1]
                best_paths[b, t] = psi[b, t + 1, current_tag_idx_at_t_plus_1]

        return best_scores, best_paths

    def forward(self, input_ids, segment_ids, input_mask):
        """Perform inference (decode the best path)."""
        mask = input_mask.float()
        feats = self._get_encoder_features(input_ids, segment_ids, input_mask)
        best_scores, best_paths = self._viterbi_decode(feats, mask)
        return best_scores, best_paths

# ## 3. Inference Functions

def load_model(checkpoint_path, hf_token=None):
    """
    Load a saved BERT-CRF model from checkpoint.
    
    Args:
        checkpoint_path: Path to the model checkpoint file (.pt)
        hf_token: Optional token for accessing gated HuggingFace models
        
    Returns:
        model, tokenizer, label_map, idx2label, max_seq_length
    """
    try:
        # Try weights_only=False first since our checkpoints have more than just weights
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    except Exception as e:
        print(f"Error with weights_only=False: {e}, trying with weights_only=True...")
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
    
    # Extract configuration from checkpoint
    model_type = checkpoint.get('model_type', checkpoint.get('config', {}).get('model_type', 'bert-base-cased'))
    label_map = checkpoint['label_map']
    idx2label = checkpoint['idx2label']
    max_seq_length = checkpoint.get('max_seq_length', 256)
    
    # Special token IDs from the label map
    start_label_id = label_map['[CLS]']
    stop_label_id = label_map['[SEP]']
    
    # Initialize model
    model = BERT_CRF_NER(
        model_type=model_type,
        start_label_id=start_label_id,
        stop_label_id=stop_label_id,
        num_labels=len(label_map),
        device=device,
        hf_token=hf_token
    )
    
    # Load weights
    model.load_state_dict(checkpoint['model_state'])
    model.to(device)
    model.eval()
    
    # Load tokenizer from the checkpoint directory
    tokenizer_path = os.path.dirname(checkpoint_path)
    try:
        tokenizer = AutoTokenizer.from_pretrained(tokenizer_path, token=hf_token)
    except Exception as e:
        print(f"Error loading tokenizer from {tokenizer_path}: {e}")
        print("Falling back to loading tokenizer from model_type")
        tokenizer = AutoTokenizer.from_pretrained(model_type, token=hf_token)
    
    print(f"Model {model_type} loaded successfully from {checkpoint_path}")
    print(f"Label map contains {len(label_map)} labels")
    
    return model, tokenizer, label_map, idx2label, max_seq_length

def predict_entities(model, tokenizer, text, label_map, idx2label, max_seq_length):
    """
    Predicts entities in a given text using the BERT-CRF model.
    
    Args:
        model: The BERT-CRF model
        tokenizer: Tokenizer corresponding to the model
        text: Input text string
        label_map: Dictionary mapping label names to IDs
        idx2label: Dictionary mapping label IDs to names
        max_seq_length: Maximum sequence length for the model
        
    Returns:
        List of detected entities with text, type, start and end token positions
    """
    model.eval()
    
    if not text or not text.strip():
        return []
    
    # Split text into words
    words = text.strip().split()
    if not words:
        return []
    
    # Create a dummy example with 'O' labels (not used for prediction)
    example = InputExample(guid="predict_0", words=words, labels=['O'] * len(words))
    
    # Convert to features
    features = example2feature(example, tokenizer, label_map, max_seq_length)
    
    # Convert to tensors
    input_ids = torch.LongTensor([features.input_ids]).to(device)
    input_mask = torch.LongTensor([features.input_mask]).to(device)
    segment_ids = torch.LongTensor([features.segment_ids]).to(device)
    predict_mask = torch.BoolTensor([features.predict_mask]).to(device)
    
    # Run model inference
    with torch.no_grad():
        _, tag_seq = model(input_ids, segment_ids, input_mask)
    
    # Process predictions
    p_mask = predict_mask[0].cpu().numpy()
    t_seq = tag_seq[0].cpu().numpy()
    
    # Align predictions with original words (only for tokens with predict_mask=1)
    valid_predictions = []
    for i, m in enumerate(p_mask):
        if m and i < len(t_seq):
            tag_id = t_seq[i]
            label = idx2label.get(tag_id, 'O')
            valid_predictions.append(label)
    
    # Handle potential mismatch due to truncation
    if len(valid_predictions) != len(words):
        words = words[:len(valid_predictions)]
    
    # Convert BIO tags to entities
    entities = []
    current_entity = None
    
    for i, (word, tag) in enumerate(zip(words, valid_predictions)):
        # Parse the tag
        if tag.startswith('B-'):
            # End any previous entity
            if current_entity:
                entities.append(current_entity)
            # Start a new entity
            entity_type = tag[2:]
            current_entity = {'text': word, 'start_token': i, 'end_token': i, 'type': entity_type}
            
        elif tag.startswith('I-'):
            # Continue an entity if types match
            entity_type = tag[2:]
            if current_entity and entity_type == current_entity['type']:
                current_entity['text'] += ' ' + word
                current_entity['end_token'] = i
            else:
                # Handle I- without matching B- (treat as B-)
                if current_entity:
                    entities.append(current_entity)
                current_entity = {'text': word, 'start_token': i, 'end_token': i, 'type': entity_type}
                
        elif tag != 'O':
            # Handle tags without BIO prefix (treat as B-)
            if current_entity:
                entities.append(current_entity)
            current_entity = {'text': word, 'start_token': i, 'end_token': i, 'type': tag}
            
        else:  # O tag
            if current_entity:
                entities.append(current_entity)
                current_entity = None
    
    # Add the final entity if it exists
    if current_entity:
        entities.append(current_entity)
    
    return entities

def visualize_entities(text, entities):
    """
    Creates a visual representation of entities in HTML.
    
    Args:
        text: Original text string
        entities: List of entity dictionaries (from predict_entities)
        
    Returns:
        HTML representation of the text with colored entity spans
    """
    # Define colors for entity types (can be extended)
    colors = {
        'Malware': '#FFB3BA',  # Light red
        'Attack': '#BAFFC9',   # Light green
        'Tool': '#BAE1FF',     # Light blue
        'System': '#FFFFBA',   # Light yellow
        'Organization': '#FFD8BA',  # Light orange
        'Person': '#DABAFA',   # Light purple
        'Location': '#B3D9C9', # Light teal
        'CVE': '#F5D0C5',      # Light salmon
        'URL': '#C1FFC1',      # Light mint
        'TTP': '#C7CEEA',      # Light periwinkle
        'File': '#E0E0E0',     # Light gray
        'Vulnerability': '#FFB6C1',  # Light pink
        'Indicator': '#B0E0E6', # Light powder blue
        'Campaign': '#D7BDE2',  # Light lavender
        'Date': '#D5D8DC',     # Light silver
        'Domain': '#A3E4D7',   # Light aqua
        'Tool_Command': '#FADBD8', # Light coral
        'Hash': '#EBDEF0'      # Light mauve
    }
    
    # Default color for unknown entity types
    default_color = '#E8E8E8'  # Light gray
    
    # Split the text into words for better handling
    words = text.split()
    
    # Create a list to hold word-level HTML spans
    html_spans = []
    
    # For each word, check if it's part of an entity
    for i, word in enumerate(words):
        # Find if this word is part of an entity
        entity_match = None
        for entity in entities:
            if entity['start_token'] <= i <= entity['end_token']:
                entity_match = entity
                break
        
        # Create appropriate HTML for this word
        if entity_match:
            entity_type = entity_match['type']
            color = colors.get(entity_type, default_color)
            html_spans.append(f'<span style="background-color: {color}; padding: 2px; border-radius: 3px;" title="{entity_type}">{word}</span>')
        else:
            html_spans.append(word)
    
    # Join the HTML spans with spaces
    html_text = ' '.join(html_spans)
    
    # Create a legend for entity types
    legend_html = '<div style="margin-top: 15px; font-size: 0.9em;"><strong>Entity Types:</strong><br>'
    
    # Get unique entity types in the current text
    unique_types = set(entity['type'] for entity in entities)
    
    # Add colored boxes for each entity type
    for entity_type in sorted(unique_types):
        color = colors.get(entity_type, default_color)
        legend_html += f'<span style="display: inline-block; margin: 2px; padding: 2px 8px; background-color: {color}; border-radius: 3px;">{entity_type}</span> '
    
    legend_html += '</div>'
    
    # Return the complete HTML
    return f'<div style="line-height: 1.5; font-size: 1.1em;">{html_text}</div>{legend_html}'

# ## 4. Set Up Model Loading & Inference

# Define default HuggingFace token (helpful for gated models like DarkBERT)
HF_TOKEN = "YOUR_HF_TOKEN_HERE"

# Default base path for models
BASE_DIR = "/home/yasir.ech-chammakhy/lustre/cyber_cc-lcbfvhtc9qm/users/yasir.ech-chammakhy/BEACON/outputs_beacon"

# Get available models in the base directory
available_models = []
if os.path.exists(BASE_DIR):
    for model_dir in os.listdir(BASE_DIR):
        model_path = os.path.join(BASE_DIR, model_dir)
        if os.path.isdir(model_path) and os.path.exists(os.path.join(model_path, "best_model.pt")):
            available_models.append(model_dir)

print(f"Found {len(available_models)} available models:")
for i, model_name in enumerate(available_models, 1):
    print(f"{i}. {model_name}")

# ## 5. Load a Model

# Choose a model to load
# You can replace this with the model you want to use
model_choice = "roberta-base_unified"  # This should be one of the models from available_models
# Or you can use an index: model_choice = available_models[0]

if model_choice not in available_models and model_choice.isdigit():
    idx = int(model_choice) - 1
    if 0 <= idx < len(available_models):
        model_choice = available_models[idx]

model_path = os.path.join(BASE_DIR, model_choice, "best_model.pt")

# Load the model
if os.path.exists(model_path):
    print(f"Loading model from {model_path}")
    model, tokenizer, label_map, idx2label, max_seq_length = load_model(model_path, HF_TOKEN)
    print("Model loaded successfully!")
else:
    print(f"Model not found at {model_path}")
    # You can provide a different path here if needed
    # model_path = "/your/custom/path/to/model.pt"
    # model, tokenizer, label_map, idx2label, max_seq_length = load_model(model_path, HF_TOKEN)

# ## 6. Run Inference on Sample Text

# Sample text for cybersecurity NER
sample_texts = [
    "The APT29 group deployed Cobalt Strike beacons after exploiting CVE-2021-44228 to gain initial access.",
    "Ransomware operators from BlackCat exfiltrated sensitive data before encrypting Windows systems with the .meow extension.",
    "CISA released an advisory about DDoS attacks targeting financial institutions using Mirai botnets.",
    "Threat actors used spear-phishing emails with malicious macros to deliver Emotet malware to healthcare organizations.",
    "The zero-day vulnerability in Chrome was exploited in the wild to deliver BazarLoader through compromised websites."
]

# Run inference on all sample texts
for i, text in enumerate(sample_texts, 1):
    print(f"\n--- Sample {i} ---")
    print(f"Text: {text}")
    
    # Predict entities
    entities = predict_entities(model, tokenizer, text, label_map, idx2label, max_seq_length)
    
    # Display detected entities
    print("\nDetected entities:")
    for entity in entities:
        print(f"  - {entity['text']} → {entity['type']} (tokens {entity['start_token']}-{entity['end_token']})")
    
    # Visualize with HTML highlighting
    display(HTML(visualize_entities(text, entities)))

# ## 7. Interactive Inference

# Function for interactive inference
def run_inference_on_text(text, model_name=None):
    """
    Run inference on custom text.
    
    Args:
        text: Text to analyze
        model_name: Optional - name of model to use (will use current model if None)
    """
    global model, tokenizer, label_map, idx2label, max_seq_length
    
    # Load a different model if specified
    if model_name and model_name != model_choice:
        new_model_path = os.path.join(BASE_DIR, model_name, "best_model.pt")
        if os.path.exists(new_model_path):
            model, tokenizer, label_map, idx2label, max_seq_length = load_model(new_model_path, HF_TOKEN)
            print(f"Switched to model: {model_name}")
        else:
            print(f"Model {model_name} not found. Using current model.")
    
    # Predict entities
    entities = predict_entities(model, tokenizer, text, label_map, idx2label, max_seq_length)
    
    # Display detected entities
    print("\nDetected entities:")
    for entity in entities:
        print(f"  - {entity['text']} → {entity['type']} (tokens {entity['start_token']}-{entity['end_token']})")
    
    # Visualize with HTML highlighting
    display(HTML(visualize_entities(text, entities)))
    
    return entities

# Enter your own text for analysis
your_text = "The threat actor APT33 deployed Cobalt Strike after exploiting CVE-2022-26809 to gain access to the network."

# Run inference on your text
detected_entities = run_inference_on_text(your_text)

# ## 8. Compare Multiple Models on the Same Text

def compare_models_on_text(text, model_names):
    """
    Compare entity detection across multiple models.
    
    Args:
        text: Text to analyze
        model_names: List of model names to compare
    """
    results = {}
    
    for model_name in model_names:
        model_path = os.path.join(BASE_DIR, model_name, "best_model.pt")
        if not os.path.exists(model_path):
            print(f"Model {model_name} not found. Skipping.")
            continue
            
        print(f"Loading and running {model_name}...")
        try:
            curr_model, curr_tokenizer, curr_label_map, curr_idx2label, curr_max_seq_length = load_model(model_path, HF_TOKEN)
            entities = predict_entities(curr_model, curr_tokenizer, text, curr_label_map, curr_idx2label, curr_max_seq_length)
            results[model_name] = entities
        except Exception as e:
            print(f"Error with model {model_name}: {e}")
            results[model_name] = []
    
    # Display results in a table
    all_entities = []
    for model_name, entities in results.items():
        for entity in entities:
            entity_dict = {
                'model': model_name,
                'text': entity['text'],
                'type': entity['type'],
                'position': f"{entity['start_token']}-{entity['end_token']}"
            }
            all_entities.append(entity_dict)
    
    # Create DataFrame
    if all_entities:
        df = pd.DataFrame(all_entities)
        display(df)
        
        # Create visual comparison
        plt.figure(figsize=(12, 6))
        model_counts = df.groupby(['model'])['type'].count()
        sns.barplot(x=model_counts.index, y=model_counts.values)
        plt.title('Number of Entities Detected by Each Model')
        plt.xlabel('Model')
        plt.ylabel('Entity Count')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
        
        # Compare entity types
        plt.figure(figsize=(14, 8))
        type_counts = df.groupby(['model', 'type']).size().reset_index(name='count')
        sns.barplot(x='type', y='count', hue='model', data=type_counts)
        plt.title('Entity Types Detected by Each Model')
        plt.xlabel('Entity Type')
        plt.ylabel('Count')
        plt.xticks(rotation=45, ha='right')
        plt.legend(title='Model')
        plt.tight_layout()
        plt.show()
        
        # Display visual comparison of entity recognition
        for model_name in results:
            print(f"\n--- Entities detected by {model_name} ---")
            entities = results[model_name]
            display(HTML(visualize_entities(text, entities)))
    else:
        print("No entities found across all models.")
    
    return results

# Select models to compare
models_to_compare = [
    "roberta-base_unified",
    "s2w-ai_DarkBERT_unified",
    "bert-base-cased_unified"
]

# Text to compare on
comparison_text = "The Chinese APT41 group utilized the ShadowPad backdoor to compromise Windows servers after exploiting CVE-2021-26855 in Exchange."

# Run the comparison
# Compare models on the text
# Uncomment the next line to run the comparison
# model_comparison = compare_models_on_text(comparison_text, models_to_compare)

# ## 9. Batch Processing

def process_text_file(file_path, model_name=None):
    """
    Process all lines in a text file and extract entities.
    
    Args:
        file_path: Path to a text file with one text example per line
        model_name: Optional model name to use for processing
    
    Returns:
        DataFrame with all detected entities
    """
    global model, tokenizer, label_map, idx2label, max_seq_length
    
    # Load a different model if specified
    if model_name:
        new_model_path = os.path.join(BASE_DIR, model_name, "best_model.pt")
        if os.path.exists(new_model_path):
            model, tokenizer, label_map, idx2label, max_seq_length = load_model(new_model_path, HF_TOKEN)
            print(f"Using model: {model_name}")
    
    all_results = []
    
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
        
        print(f"Processing {len(lines)} texts from {file_path}")
        
        for i, text in enumerate(lines):
            text = text.strip()
            if not text:
                continue
                
            entities = predict_entities(model, tokenizer, text, label_map, idx2label, max_seq_length)
            
            for entity in entities:
                all_results.append({
                    'text_id': i + 1,
                    'text': text[:100] + ('...' if len(text) > 100 else ''),
                    'entity_text': entity['text'],
                    'entity_type': entity['type'],
                    'start_token': entity['start_token'],
                    'end_token': entity['end_token']
                })
            
            # Show progress
            if (i + 1) % 10 == 0:
                print(f"Processed {i + 1}/{len(lines)} texts")
        
        # Create and return DataFrame
        results_df = pd.DataFrame(all_results)
        display(results_df)
        
        # Save results to CSV if requested
        # results_df.to_csv('ner_results.csv', index=False)
        
        # Visualize entity type distribution
        if not results_df.empty:
            plt.figure(figsize=(12, 6))
            sns.countplot(y='entity_type', data=results_df, order=results_df['entity_type'].value_counts().index)
            plt.title('Entity Type Distribution')
            plt.xlabel('Count')
            plt.ylabel('Entity Type')
            plt.tight_layout()
            plt.show()
        
        return results_df
    
    except Exception as e:
        print(f"Error processing file: {e}")
        return pd.DataFrame()

# Example usage:
# Set the path to your text file
# sample_file = "/path/to/your/texts.txt"
# process_results = process_text_file(sample_file)

# ## 10. Export Predictions to JSON/CSV

def export_predictions(text, entities, format='json', output_path=None):
    """
    Export entity predictions to JSON or CSV.
    
    Args:
        text: Original text that was analyzed
        entities: List of entity dictionaries from predict_entities
        format: 'json' or 'csv'
        output_path: Optional path to save the output file
    
    Returns:
        Path to saved file or string of content
    """
    # Prepare data
    result = {
        'text': text,
        'entities': entities
    }
    
    if format.lower() == 'json':
        # Convert to JSON
        json_data = json.dumps(result, indent=2)
        
        if output_path:
            with open(output_path, 'w', encoding='utf-8') as f:
                f.write(json_data)
            print(f"Results saved to {output_path}")
            return output_path
        else:
            return json_data
    
    elif format.lower() == 'csv':
        # Convert to DataFrame and then CSV
        if not entities:
            df = pd.DataFrame(columns=['text', 'entity_text', 'entity_type', 'start_token', 'end_token'])
        else:
            rows = []
            for entity in entities:
                rows.append({
                    'text': text,
                    'entity_text': entity['text'],
                    'entity_type': entity['type'],
                    'start_token': entity['start_token'],
                    'end_token': entity['end_token']
                })
            df = pd.DataFrame(rows)
        
        if output_path:
            df.to_csv(output_path, index=False)
            print(f"Results saved to {output_path}")
            return output_path
        else:
            return df.to_csv(index=False)
    
    else:
        print(f"Unsupported format: {format}. Use 'json' or 'csv'.")
        return None

# Example usage:
# export_predictions(your_text, detected_entities, 'json', 'predictions.json')

# ## 11. Customizable Text Input

from IPython.display import clear_output

def analyze_custom_text():
    """Interactive function to analyze custom text input with the loaded model."""
    text = input("Enter text to analyze (press Enter twice to submit):\n")
    
    # Allow multi-line input
    line = input()
    while line:
        text += "\n" + line
        line = input()
    
    clear_output()
    print(f"Analyzing text:\n{text}\n")
    
    # Run inference
    entities = predict_entities(model, tokenizer, text, label_map, idx2label, max_seq_length)
    
    # Display results
    print("\nDetected entities:")
    for entity in entities:
        print(f"  - {entity['text']} → {entity['type']} (tokens {entity['start_token']}-{entity['end_token']})")
    
    # Visualize
    display(HTML(visualize_entities(text, entities)))
    
    # Option to save results
    save_choice = input("\nDo you want to save these results? (y/n): ").lower()
    if save_choice.startswith('y'):
        format_choice = input("Choose format (json/csv): ").lower()
        if format_choice in ['json', 'csv']:
            filename = input(f"Enter filename (default: results.{format_choice}): ")
            if not filename:
                filename = f"results.{format_choice}"
            export_predictions(text, entities, format_choice, filename)
    
    return entities

# Uncomment to run the interactive analysis
# my_entities = analyze_custom_text()

# ## 12. Model Performance Analysis

def get_model_info(model_name):
    """Get training history and information about a model."""
    model_dir = os.path.join(BASE_DIR, model_name)
    
    # Try to load training history
    history_path = os.path.join(model_dir, 'training_history.json')
    history = None
    if os.path.exists(history_path):
        try:
            with open(history_path, 'r') as f:
                history = json.load(f)
            print(f"Loaded training history for {model_name}")
        except Exception as e:
            print(f"Error loading training history: {e}")
    
    # Try to load evaluation results
    eval_path = os.path.join(model_dir, 'final_evaluation_results.json')
    eval_results = None
    if os.path.exists(eval_path):
        try:
            with open(eval_path, 'r') as f:
                eval_results = json.load(f)
            print(f"Loaded evaluation results for {model_name}")
        except Exception as e:
            print(f"Error loading evaluation results: {e}")
    
    # Plot training metrics if available
    if history and 'train_loss' in history and 'valid_f1' in history:
        plt.figure(figsize=(12, 5))
        
        # Plot training loss
        plt.subplot(1, 2, 1)
        plt.plot(history['train_loss'])
        plt.title('Training Loss')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.grid(True)
        
        # Plot validation F1
        plt.subplot(1, 2, 2)
        plt.plot(history['valid_f1'])
        plt.title('Validation F1 Score')
        plt.xlabel('Epoch')
        plt.ylabel('F1')
        plt.grid(True)
        
        plt.tight_layout()
        plt.show()
    
    # Display evaluation metrics if available
    if eval_results:
        # Overall metrics
        print("\nOverall Test F1 Score:", eval_results.get('overall_test_f1', 'N/A'))
        print("Best Validation F1 Score:", eval_results.get('best_valid_f1', 'N/A'))
        print("Best Epoch:", eval_results.get('best_epoch', 'N/A'))
        
        # Source-specific results
        source_results = eval_results.get('source_specific_test_results', {})
        if source_results:
            print("\nSource-Specific Results:")
            sources_df = pd.DataFrame({
                'Source': list(source_results.keys()),
                'F1 Score': [source_data.get('f1', 0) for source_data in source_results.values()]
            })
            display(sources_df)
            
            # Plot source-specific F1 scores
            plt.figure(figsize=(10, 5))
            sns.barplot(x='Source', y='F1 Score', data=sources_df)
            plt.title(f'F1 Scores by Source ({model_name})')
            plt.xlabel('Source')
            plt.ylabel('F1 Score')
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()
        
        # Per-class metrics
        class_metrics = eval_results.get('overall_class_metrics', {})
        if class_metrics:
            print("\nTop Entity Class Performance:")
            classes_data = []
            for cls_name, cls_metrics in class_metrics.items():
                # Skip special tokens
                if cls_name in ['O', 'X', '[CLS]', '[SEP]']:
                    continue
                classes_data.append({
                    'Class': cls_name,
                    'Precision': cls_metrics.get('precision', 0),
                    'Recall': cls_metrics.get('recall', 0),
                    'F1': cls_metrics.get('f1', 0),
                    'Support': cls_metrics.get('support', 0)
                })
            
            classes_df = pd.DataFrame(classes_data)
            if not classes_df.empty:
                classes_df = classes_df.sort_values('F1', ascending=False)
                display(classes_df.head(10))  # Show top 10 classes by F1
                
                # Plot top 10 classes by F1 score
                top_classes = classes_df.head(10)
                plt.figure(figsize=(12, 6))
                plt.subplot(1, 2, 1)
                sns.barplot(x='F1', y='Class', data=top_classes)
                plt.title('Top 10 Classes by F1 Score')
                plt.xlabel('F1 Score')
                
                # Plot distribution of support
                plt.subplot(1, 2, 2)
                sns.barplot(x='Support', y='Class', data=top_classes)
                plt.title('Sample Support for Top Classes')
                plt.xlabel('Number of Examples')
                
                plt.tight_layout()
                plt.show()
    
    # If no history or results
    if not history and not eval_results:
        print(f"No training history or evaluation results found for {model_name}")
    
    return history, eval_results

# Example usage:
# model_name = "roberta-base_unified"
# model_history, model_eval_results = get_model_info(model_name)

# ## 13. Conclusion and Next Steps

print("""
# Conclusion

This notebook provides a complete solution for running inference with your trained BEACON models. You can:

1. **Load any of your trained models** to perform inference
2. **Run predictions on individual texts** or batch process multiple texts
3. **Visualize entity predictions** with colored highlighting
4. **Compare performance across different models**
5. **Analyze model training history and evaluation metrics**
6. **Export predictions** to JSON or CSV for further analysis

# Next Steps

- Fine-tune the model on domain-specific data for your particular use case
- Implement this inference pipeline in a production environment
- Create an interactive API or web interface for your model
- Evaluate the model on new datasets to assess generalization
- Combine model outputs with other security tools in your workflow

Feel free to modify and extend this notebook for your specific needs!
""")

2025-05-01 12:20:27.821129: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746098428.152215  203296 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746098428.243851  203296 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-01 12:20:29.206222: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Using device: cuda
Found 5 available models:
1. s2w-ai_DarkBERT_unified
2. ehsanaghaei_SecureBERT_unified
3. bert-base-cased_unified
4. roberta-base_unified
5. markusbayer_CySecBERT_unified
Loading model from /home/yasir.ech-chammakhy/lustre/cyber_cc-lcbfvhtc9qm/users/yasir.ech-chammakhy/BEACON/outputs_beacon/roberta-base_unified/best_model.pt
Initializing encoder: roberta-base


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model roberta-base loaded successfully from /home/yasir.ech-chammakhy/lustre/cyber_cc-lcbfvhtc9qm/users/yasir.ech-chammakhy/BEACON/outputs_beacon/roberta-base_unified/best_model.pt
Label map contains 46 labels
Model loaded successfully!

--- Sample 1 ---
Text: The APT29 group deployed Cobalt Strike beacons after exploiting CVE-2021-44228 to gain initial access.

Detected entities:
  - APT29 → Threat-Actor (tokens 1-1)
  - Cobalt Strike → Tool (tokens 4-5)
  - CVE-2021-44228 → Vulnerability (tokens 9-9)



--- Sample 2 ---
Text: Ransomware operators from BlackCat exfiltrated sensitive data before encrypting Windows systems with the .meow extension.

Detected entities:
  - BlackCat → Threat-Actor (tokens 3-3)
  - Windows → Software (tokens 9-9)
  - .meow → File (tokens 13-13)



--- Sample 3 ---
Text: CISA released an advisory about DDoS attacks targeting financial institutions using Mirai botnets.

Detected entities:
  - CISA → Identity (tokens 0-0)
  - DDoS attacks → Intrusion-Set (tokens 5-6)
  - financial institutions → Identity (tokens 8-9)
  - Mirai botnets. → Tool (tokens 11-12)



--- Sample 4 ---
Text: Threat actors used spear-phishing emails with malicious macros to deliver Emotet malware to healthcare organizations.

Detected entities:
  - actors → Threat-Actor (tokens 1-1)
  - spear-phishing emails → Attack-Pattern (tokens 3-4)
  - Emotet malware → Tool (tokens 10-11)
  - healthcare organizations. → Identity (tokens 13-14)



--- Sample 5 ---
Text: The zero-day vulnerability in Chrome was exploited in the wild to deliver BazarLoader through compromised websites.

Detected entities:
  - zero-day vulnerability → Vulnerability (tokens 1-2)
  - Chrome → Tool (tokens 4-4)
  - BazarLoader → Malware (tokens 12-12)



Detected entities:
  - threat actor → Threat-Actor (tokens 1-2)
  - APT33 → Threat-Actor (tokens 3-3)
  - Cobalt Strike → Tool (tokens 5-6)
  - CVE-2022-26809 → Vulnerability (tokens 9-9)



# Conclusion

This notebook provides a complete solution for running inference with your trained BEACON models. You can:

1. **Load any of your trained models** to perform inference
2. **Run predictions on individual texts** or batch process multiple texts
3. **Visualize entity predictions** with colored highlighting
4. **Compare performance across different models**
5. **Analyze model training history and evaluation metrics**
6. **Export predictions** to JSON or CSV for further analysis

# Next Steps

- Fine-tune the model on domain-specific data for your particular use case
- Implement this inference pipeline in a production environment
- Create an interactive API or web interface for your model
- Evaluate the model on new datasets to assess generalization
- Combine model outputs with other security tools in your workflow

Feel free to modify and extend this notebook for your specific needs!

